In [1]:
print(34)

34


In [2]:
from dotenv import load_dotenv
load_dotenv()
import os

In [5]:
GATEWAY_CONFIG = {
    "strategy": {"mode": "fallback"},
    "cache": {"mode": "simple"},
    "retry": {
        "attempts": 2,
        "on_status_codes": [429, 503]
    },
    "targets": [
        {"override_params": {"model": f"@rag/llama-3.3-70b-versatile"}},
        {"override_params": {"model": f"@brag/llama-3.1-8b-instant"}},
    ]
}

In [7]:

from portkey_ai import Portkey, createHeaders, PORTKEY_GATEWAY_URL
from langchain_openai import ChatOpenAI
import time
import os
import sys


# Production gateway config:
#   - Fallback: primary @rag/llama-3.3-70b-versatile → @brag/llama-3.1-8b-instant on failure
#   - Cache: semantic mode (requires Portkey Enterprise — silently falls back to simple on free/starter)
#   - Retry: 2 attempts on rate limit / server error before triggering the fallback target
GROQ_SLUG    =  "rag"  # your Groq integration slug
GROQ_MODEL   = f"@{GROQ_SLUG}/llama-3.3-70b-versatile"

# Second Groq integration for multi-provider experiments (5, 6, 10)
# Uses a smaller/faster model as the fallback target
GROQ_SLUG_2      =  "brag"           # your second Groq slug
GROQ_MODEL_SMALL = f"@{GROQ_SLUG_2}/llama-3.1-8b-instant"

GATEWAY_CONFIG = {
    "strategy": {"mode": "fallback"},
    "cache": {"mode": "simple"},
    "retry": {
        "attempts": 2,
        "on_status_codes": [429, 503]
    },
    "targets": [
        {"override_params": {"model": f"@{GROQ_MODEL}/llama-3.3-70b-versatile"}},
        {"override_params": {"model": f"@{GROQ_MODEL_SMALL}/llama-3.1-8b-instant"}},
    ]
}
PORTKEY_GATEWAY_URL=os.getenv("PORTKEY_API_KEY")
print(f"PORTKEY_GATEWAY_URL: {PORTKEY_GATEWAY_URL} to validate that the environment variable is set correctly")
os.environ["PORTKEY_GATEWAY_URL"] = PORTKEY_GATEWAY_URL

portkey_fallback = Portkey(api_key=PORTKEY_GATEWAY_URL, config=GATEWAY_CONFIG)


print(f"Strategy: {GROQ_MODEL} → {GROQ_MODEL_SMALL} on failure\n")

fallback_questions = [
    "What is Intel QuickAssist Technology?",
    "Explain Kubernetes persistent volume claims.",
]

for q in fallback_questions:
    try:
        t0 = time.time()
        r = portkey_fallback.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        show(q, r.choices[0].message.content, ms, label="fallback ready")
    except Exception as e:
        print(f"❌ {e}")
        print("   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04")

print("\nFallback chain:")
print("  Groq 2xx   → return immediately")
print("  Groq 4xx/5xx → try small Groq model (8b) automatically")
print("  Check Portkey Logs to see fallback activations")

PORTKEY_GATEWAY_URL: D84c7B0/2jWGZAylZk27ZxHr6GF1 to validate that the environment variable is set correctly
Strategy: @rag/llama-3.3-70b-versatile → @brag/llama-3.1-8b-instant on failure

❌ Error code: 400 - {'status': 'failure', 'error': {'code': 'inline_config_blocked', 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead.", 'field': 'x-portkey-config'}, 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead."}
   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04
❌ Error code: 400 - {'status': 'failure', 'error': {'code': 'inline_config_blocked', 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead.", 'field': 'x-portkey-config'}, 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config b

In [ ]:
groq_slug = "rag"  # or "brag" for the smaller model
fall_back_slug = "brag"  # or "rag" for the larger model


In [ ]:
import os
import time
import uuid
import json
from dotenv import load_dotenv
from portkey_ai import Portkey, createHeaders, PORTKEY_GATEWAY_URL

load_dotenv(dotenv_path="../.env")

# Portkey API key — authenticates your app to the Portkey gateway
PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")
print(f"PORTKEY_API_KEY: {PORTKEY_API_KEY}")

# Provider slug — the name you gave when adding Groq in the Portkey dashboard
# Model format: @<slug>/<model-name>
GROQ_SLUG    =  "rag"  # your Groq integration slug
GROQ_MODEL   = f"@{GROQ_SLUG}/llama-3.3-70b-versatile"

# Second Groq integration for multi-provider experiments (5, 6, 10)
# Uses a smaller/faster model as the fallback target
GROQ_SLUG_2      =  "brag"           # your second Groq slug
GROQ_MODEL_SMALL = f"@{GROQ_SLUG_2}/llama-3.1-8b-instant"